# Tutorial 3: Structure-Property Mapping

### Goal:
- Visualize chemical data in low dimensions
- Compare PCA vs PCovR
- Understand structure–property relationships
- Explore data interactively with chemiscope

In [ ]:
import numpy as np
from skmatter.datasets import load_csd_1000r
from skmatter.preprocessing import StandardFlexibleScaler

X, y = load_csd_1000r(
    return_X_y=True,
)

X = StandardFlexibleScaler(column_wise=False).fit_transform(X)
y = StandardFlexibleScaler(column_wise=True).fit_transform(y)
print("X shape:", X.shape)
print("y shape:", y.shape)

## Structure–Property Mapping

We want a low-dimensional representation $T$.

Two approaches:

- **PCA** → preserves variance (unsupervised)
- **PCovR** → preserves variance + predicts property (supervised)

PCovR introduces a tradeoff:

$$\ell = \alpha \ell(T \rightarrow X) + (1-\alpha)\ell(T \rightarrow y)$$

- $(\alpha = 1)$: PCA-like (structure only)  
- $(\alpha = 0)$: regression-focused (property only)

In [ ]:
from sklearn.decomposition import PCA

T_pca = PCA(n_components=2).fit_transform(X)

In [ ]:
from skmatter.decomposition import PCovR

pcovr = PCovR(mixing=0.5, n_components=2)
T_pcovr = pcovr.fit_transform(X, y)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

sc1 = axes[0].scatter(T_pca[:, 0], T_pca[:, 1], c=y)
axes[0].set_title("PCA")

sc2 = axes[1].scatter(T_pcovr[:, 0], T_pcovr[:, 1], c=y)
axes[1].set_title("PCovR")

for ax in axes:
    ax.set_xlabel("Component 1")
    ax.set_ylabel("Component 2")

plt.colorbar(sc2, label="Target")
plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# TODO: change alpha
# ==========================================

# Try:
# alpha = 0.0, 0.5, 1.0

# pcovr = PCovR(mixing=___, n_components=2)
# T_new = pcovr.fit_transform(X, y)

# Plot or visualize

In [ ]:
import matplotlib.pyplot as plt
from skmatter.decomposition import PCovR

alphas = np.linspace(0, 1, 10)

reconstruction_losses = []
prediction_losses = []
total_losses = []

for alpha in alphas:
    pcovr = PCovR(mixing=alpha, n_components=2)
    T = pcovr.fit_transform(X, y)

    # Reconstruction of X
    X_rec = pcovr.inverse_transform(T)
    recon_loss = np.mean((X - X_rec) ** 2)

    # Prediction of y
    y_pred = pcovr.predict(X)
    pred_loss = np.mean((y - y_pred) ** 2)

    total_loss = alpha * recon_loss + (1 - alpha) * pred_loss

    reconstruction_losses.append(recon_loss)
    prediction_losses.append(pred_loss)
    total_losses.append(total_loss)

plt.figure(figsize=(6, 4))

plt.plot(alphas, reconstruction_losses, label="Reconstruction loss")
plt.plot(alphas, prediction_losses, label="Prediction loss")
plt.plot(alphas, total_losses, label="Total loss", linewidth=2)

plt.xlabel("alpha")
plt.ylabel("Loss")
plt.title("PCovR Loss vs Alpha")

plt.legend()
plt.grid(alpha=0.3)

plt.show()

### Takeaway

PCovR does not explain *a prediction*—

→ it explains the **organization of chemical space**

## Interactive Exploration (External)

To explore structures interactively, visit:

https://archive.materialscloud.org/records/gxcgf-rkh55

---

### What to do there:

- Explore maps in 3D  
- Look for structural similarities  
- Connect structure to properties  